# Predicting Irrigation Need

Link to Competittion: https://www.kaggle.com/competitions/playground-series-s6e4/overview

## Imports

In [1]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.figsize'] = (12, 6)

import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

import xgboost as xgb
import optuna

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, accuracy_score, confusion_matrix, mean_squared_error, classification_report, balanced_accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold, StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.cluster import KMeans
from sklearn.utils.class_weight import compute_sample_weight

from common import *

In [2]:
from platform import python_version
print('python: ', python_version())
print('pandas: ', pd.__version__)
print('numpy: ', np.__version__)
import matplotlib
print('matplotlib: ', matplotlib.__version__)
print('seaborn: ', sns.__version__)
import sklearn
print('sklearn: ', sklearn.__version__)
print('xgboost: ', xgb.__version__)

python:  3.13.11
pandas:  2.3.3
numpy:  2.3.5
matplotlib:  3.10.8
seaborn:  0.13.2
sklearn:  1.8.0
xgboost:  3.2.0


## Helpers

## Load data

In [3]:
train_df = pd.read_csv('archive/train.csv')
test_df = pd.read_csv('archive/test.csv')
orig_df = pd.read_csv('archive/irrigation_prediction.csv')

## Call the pipeline

In [4]:
df = (train_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

## Concat original dataset

In [5]:
orig_df_cleaned = (orig_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [6]:
df = pd.concat([df, orig_df_cleaned])

## Features

In [7]:
target = get_target()

In [8]:
features = get_features(df)

In [9]:
features

['soil_type',
 'soil_ph',
 'soil_moisture',
 'organic_carbon',
 'electrical_conductivity',
 'temperature_c',
 'humidity',
 'rainfall_mm',
 'sunlight_hours',
 'wind_speed_kmh',
 'crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'field_area_hectare',
 'mulching_used',
 'previous_irrigation_mm',
 'region',
 'feels_like']

In [10]:
categorical_features = [
    'crop_type',
    'crop_growth_stage',
    'season',
    'irrigation_type',
    'water_source',
    'soil_type',
    'region'
]

In [11]:
numerical_features = [f for f in features if f not in categorical_features]

In [12]:
categorical_features

['crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'soil_type',
 'region']

In [13]:
numerical_features

['soil_ph',
 'soil_moisture',
 'organic_carbon',
 'electrical_conductivity',
 'temperature_c',
 'humidity',
 'rainfall_mm',
 'sunlight_hours',
 'wind_speed_kmh',
 'field_area_hectare',
 'mulching_used',
 'previous_irrigation_mm',
 'feels_like']

In [14]:
df['mulching_used'] = df['mulching_used'].map({'Yes': 1, 'No': 0})

In [15]:
df[features]

,soil_type,soil_ph,soil_moisture,organic_carbon,electrical_conductivity,temperature_c,humidity,rainfall_mm,sunlight_hours,wind_speed_kmh,crop_type,crop_growth_stage,season,irrigation_type,water_source,field_area_hectare,mulching_used,previous_irrigation_mm,region,feels_like
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,0,112.16,East,759.6561
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,1,47.16,South,1555.3512
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,1,110.38,North,2487.1734
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,1,53.85,South,820.1124
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,0,93.19,South,1842.2442
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Silt,7.01,26.67,0.86,0.76,27.61,52.20,1075.12,7.41,19.66,Sugarcane,Sowing,Kharif,Drip,Groundwater,2.62,1,92.44,South,1441.2420
9996,Clay,5.40,49.44,0.90,1.19,34.03,52.31,1591.84,9.86,5.66,Maize,Sowing,Kharif,Rainfed,Groundwater,4.87,0,15.46,South,1780.1093
9997,Loamy,4.97,60.63,0.99,1.30,36.68,68.16,2384.87,10.75,13.40,Potato,Harvest,Kharif,Canal,Groundwater,10.08,1,116.36,North,2500.1088
9998,Loamy,7.12,44.33,1.56,1.08,31.50,64.83,2397.01,4.03,3.05,Sugarcane,Harvest,Kharif,Rainfed,Reservoir,11.11,1,118.17,East,2042.1450


In [16]:
df[target] = df[target].map({'Low': 0, 'Medium': 1, 'High': 2})

In [17]:
df[categorical_features] = df[categorical_features].astype('category')

In [18]:
categorical_features

['crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'soil_type',
 'region']

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 640000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   soil_type                640000 non-null  category
 1   soil_ph                  640000 non-null  float64 
 2   soil_moisture            640000 non-null  float64 
 3   organic_carbon           640000 non-null  float64 
 4   electrical_conductivity  640000 non-null  float64 
 5   temperature_c            640000 non-null  float64 
 6   humidity                 640000 non-null  float64 
 7   rainfall_mm              640000 non-null  float64 
 8   sunlight_hours           640000 non-null  float64 
 9   wind_speed_kmh           640000 non-null  float64 
 10  crop_type                640000 non-null  category
 11  crop_growth_stage        640000 non-null  category
 12  season                   640000 non-null  category
 13  irrigation_type          640000 non-null  category


## Stratified KFold Loop

In [20]:
xgb_params = {
    'n_jobs': -1,
    'enable_categorical': True,
    'random_state': 123
}

In [21]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)
scores = []

X, y = df[features], df[target]

In [22]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    fold_sample_weights = compute_sample_weight('balanced', y_train)
    model.fit(X_train, y_train, sample_weight=fold_sample_weights)

    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    scores.append(score)
    print(f"Fold {fold}:  {score:.5f}")

print(f"Mean: {np.mean(scores):.5f} ± {np.std(scores):.5f}")

Fold 0:  0.97205
Fold 1:  0.96528
Fold 2:  0.96949
Fold 3:  0.97057
Fold 4:  0.96972
Fold 5:  0.97087
Fold 6:  0.97243
Fold 7:  0.97461
Fold 8:  0.97185
Fold 9:  0.96717
Mean: 0.97040 ± 0.00255


## Optuna Hyperparameter Tuning

In [ ]:
def objective(trial):
    params = {
        'n_jobs': -1,
        'enable_categorical': True,
        'random_state': 123,
        'n_estimators': 2000,
        'eval_metric': 'mlogloss',
        'early_stopping_rounds': 50,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 5.0, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 5.0, log=True),
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

        sw_tr = compute_sample_weight('balanced', y_tr)
        sw_val = compute_sample_weight('balanced', y_val)

        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            sample_weight=sw_tr,
            eval_set=[(X_val, y_val)],
            sample_weight_eval_set=[sw_val],
            verbose=False,
        )

        preds = model.predict(X_val)
        score = balanced_accuracy_score(y_val, preds)
        scores.append(score)

        trial.report(np.mean(scores), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return np.mean(scores)


study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2, n_startup_trials=5),
    sampler=optuna.samplers.TPESampler(seed=123),
)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"Best CV: {study.best_value:.5f}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-04-30 21:37:38,969] A new study created in memory with name: no-name-f7451561-4e51-46de-b88a-78b11a7a270c


  0%|          | 0/30 [00:00<?, ?it/s]

## Re-validate Best Params on 10-fold

In [ ]:
best_params = {
    'n_jobs': -1,
    'enable_categorical': True,
    'random_state': 123,
    'n_estimators': 2000,
    'eval_metric': 'mlogloss',
    'early_stopping_rounds': 50,
    **study.best_params,
}

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=123)
scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    sw_tr = compute_sample_weight('balanced', y_tr)
    sw_val = compute_sample_weight('balanced', y_val)

    model = xgb.XGBClassifier(**best_params)
    model.fit(X_tr, y_tr, sample_weight=sw_tr,
              eval_set=[(X_val, y_val)], sample_weight_eval_set=[sw_val], verbose=False)

    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    scores.append(score)
    print(f"Fold {fold}: {score:.5f}")

print(f"Mean: {np.mean(scores):.5f} ± {np.std(scores):.5f}")

## Final Model

In [ ]:
final_params = {
    'n_jobs': -1,
    'enable_categorical': True,
    'random_state': 123,
    **study.best_params,
}

xgb_final_model = XGBClassifier(**final_params)
final_sample_weights = compute_sample_weight('balanced', df[target])
xgb_final_model.fit(df[features], df[target], sample_weight=final_sample_weights)

In [ ]:
df = (test_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [ ]:
df['mulching_used'] = df['mulching_used'].map({'Yes': 1, 'No': 0})

In [ ]:
df[categorical_features] = df[categorical_features].astype('category')

In [ ]:
df

In [ ]:
preds = xgb_final_model.predict(df[features])

In [ ]:
preds

In [ ]:
submission_df = pd.DataFrame(test_df['id'].copy())

In [ ]:
submission_df['Irrigation_Need'] = preds

In [ ]:
submission_df

In [ ]:
submission_df['Irrigation_Need'] = submission_df['Irrigation_Need'].map({0: 'Low', 1: 'Medium', 2: 'High'})

In [ ]:
submission_df['Irrigation_Need'].value_counts()

In [ ]:
last_submission = pd.read_csv(find_last_submission_file())

In [ ]:
last_submission['Irrigation_Need'].value_counts()

In [ ]:
# if np.allclose(last_submission['Irrigation_Need'], submission_df['Irrigation_Need']):
if all(last_submission['Irrigation_Need'].value_counts() == submission_df['Irrigation_Need'].value_counts()):
    # they are the same, don't same
    print('skipping save')
else:
    submission_df.to_csv(find_next_submission_file(), index=False)
    print('saving file')